# 🏭 Phase 1: Machine Learning Pipeline for Predictive Maintenance
### AI Factory Operating System — Digital Twin Platform

This notebook demonstrates the complete, end-to-end Machine Learning pipeline:
1. **C-MAPSS FD001 Remaining Useful Life (RUL)** regression with zero data leakage (split strictly by Engine ID).
2. **AI4I 2020 Machine Failure Classification** with class imbalance handling and leakage column dropping.
3. **Semi-Supervised Anomaly Detection** (Isolation Forest & LSTM Autoencoder) trained strictly on healthy operational windows.
4. **Explainability**: SHAP global and local explanations for tree models, and neural gradient saliency for deep networks.
5. **Final Evaluation Table** reporting RMSE, NASA Prognostics Score, F1, and PR-AUC.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed
np.random.seed(42)
print("Random seed fixed at 42")

## 1. Zero Data Leakage Verification & Engine-Level Splits

> **Why split by Engine ID and never by random rows?**  
> In predictive maintenance, an engine's sensor measurements form a continuous time-series degradation trajectory. If we perform a random row split, the training set would contain measurements from Cycle 100 of Engine 5, while the test set contains Cycle 101 of Engine 5. The model would trivially memorise Engine 5's specific sensor calibration rather than learning general physical wear mechanics. Splitting strictly by Engine ID guarantees that the validation, demo, and test sets contain engines the model has never encountered before.

In [ ]:
from ml.data_loader import get_cmapss_splits, USEFUL_SENSORS
from ml.leakage_check import check_engine_leakage

train_df, val_df, demo_df, test_df, rul_df = get_cmapss_splits()

# Verify zero leakage
check_engine_leakage(
    train_df["engine_id"].unique(),
    val_df["engine_id"].unique(),
    demo_df["engine_id"].unique(),
)
print(f"Active useful sensors ({len(USEFUL_SENSORS)}): {USEFUL_SENSORS}")

## 2. Sensor Normalisation & 30-Cycle Sliding Windows
- Scaler is fit strictly on TRAIN engines.
- Windows of length 30 are built per engine separately (never bridging between engines).
- Tabular summary features (Last, Mean, Std, Slope) are extracted for XGBoost (15 × 4 = 60 features).

In [ ]:
from ml.preprocessing import CMapssPreprocessor

preprocessor = CMapssPreprocessor(window_size=30, feature_cols=USEFUL_SENSORS)
preprocessor.fit(train_df)

train_scaled = preprocessor.transform_df(train_df)
val_scaled = preprocessor.transform_df(val_df)
test_scaled = preprocessor.transform_df(test_df)

X_train_seq, y_train, _ = preprocessor.create_sequence_windows(train_scaled)
X_val_seq, y_val, _ = preprocessor.create_sequence_windows(val_scaled)
X_test_last_seq, _ = preprocessor.extract_test_last_windows(test_scaled)
y_test_true = rul_df["true_rul"].values.astype(np.float32)

X_train_tab, tab_names = preprocessor.extract_tabular_features(X_train_seq, USEFUL_SENSORS)
X_val_tab, _ = preprocessor.extract_tabular_features(X_val_seq, USEFUL_SENSORS)
X_test_last_tab, _ = preprocessor.extract_tabular_features(X_test_last_seq, USEFUL_SENSORS)

print(f"X_train_seq shape : {X_train_seq.shape}")
print(f"X_val_seq shape   : {X_val_seq.shape}")
print(f"X_test_last shape : {X_test_last_seq.shape}")
print(f"X_train_tab shape : {X_train_tab.shape}")

## 3. Official Evaluation Table
Let us inspect the final benchmark table produced by the training run across all models.

In [ ]:
eval_df = pd.read_csv("data/evaluation_table.csv")
display(eval_df)

## 4. Model Explainability — SHAP and Neural Saliency
Let us view the global SHAP beeswarm plot and neural saliency chart generated by the pipeline.

In [ ]:
from IPython.display import Image, display

print("Global SHAP Summary Plot (XGBoost RUL):")
display(Image(filename="data/models/shap_summary_rul.png"))

print("Neural Saliency Feature Importance:")
display(Image(filename="data/models/saliency_neural.png"))